# Example: 2D $\phi^4$ Lattice Field Theory
## 1. System Setup

We consider the 2D lattice $\phi^4$ model on an $L \times L$ square lattice with periodic boundary conditions:

$$
H(\phi) = \sum_i (\phi_i^2 - 1)^2
+ J \sum_{\langle i,j \rangle} (\phi_i - \phi_j)^2
- h \sum_i \phi_i.
$$

In this notebook we focus on the zero-field case

$$
h = 0,
$$

and use a pretrained 1D diffusion prior for the on-site double-well factor

$$
p_0(\phi) \propto \exp\!\left[-\sum_i (\phi_i^2 - 1)^2\right].
$$

So GG-PA only needs to supply the nearest-neighbour coupling term through the context.

## 2. Annealed Context and the Match Condition

Let $\psi$ be the noisy field (auxiliary field), with VP forward process

$$
q_t(\psi \mid \phi) = \mathcal{N}(\psi; \alpha_t \phi, \sigma_t^2 I).
$$

For a periodic lattice, the coupling term is diagonal in Fourier space. If $\lambda_k$ denotes the PBC Laplacian eigenvalue of mode $k$, then the Gaussian context can be written as

$$
p_{\mathrm{ctx},t}(\psi)
\propto
\exp\!\left[
-\frac12 \sum_{k \neq 0} q_k(t) \, |\hat\psi_k|^2
+ \hat b_0 \, \hat\psi_0
\right],
$$

with

$$
q_k(t) = \frac{2J\lambda_k}{\alpha_t^2 - 2J\lambda_k\sigma_t^2},
\qquad
q_0(t)=0.
$$

For a nonzero external field one only needs a zero-mode shift,

$$
\hat b_0 = \frac{hL}{\alpha_t},
$$

but in this quick run we set $h=0$, so only the quadratic Fourier-space penalty remains.

This choice is made so that integrating out $\psi$ reproduces the physical lattice coupling:

$$
\int d\psi\; q_t(\psi \mid \phi) \, p_{\mathrm{ctx},t}(\psi)
\propto
\exp\!\left[-J \sum_{\langle i,j \rangle} (\phi_i - \phi_j)^2 + h \sum_i \phi_i \right].
$$

## 3. Critical Condition

The Gaussian construction is valid only if every nonzero Fourier mode has positive denominator:

$$
\alpha_t^2 - 2J\lambda_k\sigma_t^2 > 0 \qquad \text{for all } k \neq 0.
$$

In 2D, the largest Laplacian eigenvalue is $\lambda_{\max}=8$, so a sufficient condition is

$$
\alpha_t^2 - 16J\sigma_t^2 > 0.
$$

Equivalently, since $\alpha_t^2 = \bar\alpha_t$ and $\sigma_t^2 = 1-\bar\alpha_t$,

$$
\bar\alpha_t > \frac{16J}{1+16J}.
$$

The code below checks this condition for the chosen scan range.

## 4. Implementation in This Notebook

We use the system-specific adapters in `src/ggpa/systems/phi4.py`:

- `LatticeDiffusionClient`: denoise $\psi \to \phi$ with the pretrained on-site prior
- `GaussianPBCContext`: build the annealed Gaussian context in Fourier space
- `FFTGaussianAggregator`: sample $\psi \mid \phi$ exactly by FFT mode-by-mode Gaussian updates

This notebook is organized into two lightweight GG-PA demos:

- an `L = 32` final-frame snapshot section at `J = 0.10, 0.436, 0.80`
- an `L = 8` scan of the order parameter and susceptibility versus `J`

Both use the same zero-field setup `h = 0` and the same fixed diffusion time `t = 0.10`.

In the scan section we record the signed magnetization

$$
m = \frac{1}{L^2} \sum_i \phi_i,
$$

and report the quick-run observables

$$
\langle |m| \rangle,
\qquad
\chi = L^2\bigl(\langle m^2 \rangle - \langle |m| \rangle^2\bigr).
$$

The `L=32` snapshots are only illustrative, and the `L=8` scan should still be viewed as a fast qualitative phase-transition scan rather than a precision finite-size study.

In [ ]:
import sys
import time
import warnings
import logging
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'src').exists() or not (ROOT / 'checkpoints').exists():
    raise RuntimeError('Run this notebook from the project root or from the notebooks/ directory.')
SRC_PATH = ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import matplotlib.pyplot as plt
import torch

from ggpa import FixedDiffusionTimeKernel, State
from ggpa.models import SimpleDiffusion
from ggpa.systems.phi4 import (
    check_max_t_diff,
    LatticeVPForwardProcess,
    LatticeDiffusionClient,
    GaussianPBCContext,
    FFTGaussianAggregator,
    integrated_autocorrelation_time,
)

warnings.filterwarnings('ignore', category=UserWarning, module=r'torch\.cuda')
logging.getLogger('ggpa').setLevel(logging.WARNING)
plt.style.use('seaborn-v0_8-whitegrid')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

In [ ]:
ckpt_path = ROOT / 'checkpoints' / 'phi4_prior.pt'
diffusion = SimpleDiffusion.load_from_checkpoint(ckpt_path, device=device)
diffusion.eval()
diffusion = torch.compile(diffusion, mode='reduce-overhead')


# directly sample from the checkpoint
num_samples = 2000
samples = diffusion.sample(num_samples,device=device).cpu().numpy()

# prior: p(phi) is a double-well
import seaborn as sns
fig, ax = plt.subplots(figsize=(6, 5))
sns.kdeplot(samples.flatten(), ax=ax, fill=True)
ax.set_title(r'Prior Distribution of $\phi$')
ax.set_xlabel(r'$\phi$')
ax.set_ylabel('Density')
plt.show()

In [ ]:

L = 8
h = 0.0
t = 0.10

J_coarse = np.linspace(0.10, 0.80, 8)
J_dense = np.linspace(0.34, 0.54, 5)
J_values = np.sort(np.unique(np.round(np.concatenate([J_coarse, J_dense]), 3)))

N_SWEEPS = 1000
BURNIN = 250
RECORD_INTERVAL = 5
SEED_BASE = 123
CRITICAL_J_WINDOW = (0.34, 0.54)
CRITICAL_N_SWEEPS = 2000

ns = diffusion.noise_scheduler
fwd = LatticeVPForwardProcess(ns)
alpha_t = float(fwd.alpha(t))
sigma_t = float(fwd.sigma(t))
J_max_for_t = (alpha_t ** 2) / (16.0 * sigma_t ** 2)
scan_info = check_max_t_diff(ns, float(J_values.max()))

if scan_info['t_max'] is None or t >= scan_info['t_max']:
    raise RuntimeError(
        f'Chosen t={t:.3f} is outside the feasible Gaussian-context range for J_max={float(J_values.max()):.3f}. '
        f'Max allowed t is {scan_info["t_max"]}. '
        'Choose a smaller t or a smaller J range.'
    )

print('Checkpoint loaded from:', str(ckpt_path))
print(f'L={L}, h={h:.1f}, t={t:.2f}')
print(f'J grid ({len(J_values)} points): {J_values}')
print(f'default sweeps per J = {N_SWEEPS}, default burn-in = {BURNIN}, record interval = {RECORD_INTERVAL}')
print(f'critical-region window = [{CRITICAL_J_WINDOW[0]:.2f}, {CRITICAL_J_WINDOW[1]:.2f}] with {CRITICAL_N_SWEEPS} sweeps')
print(f'alpha_t = {alpha_t:.4f}, sigma_t = {sigma_t:.4f}')
print(f'Feasibility bound at this t: J < {J_max_for_t:.3f}')
print(f'check_max_t_diff(J_max={float(J_values.max()):.3f}) -> t_max = {scan_info["t_max"]:.4f}')

In [3]:
def compute_observables_from_history(magnetizations, L):
    """Compute quick-run observables from the signed magnetization series."""
    m = np.asarray(magnetizations, dtype=np.float64)
    abs_m = np.abs(m)

    mean_abs_m = np.mean(abs_m)
    m2 = np.mean(m ** 2)
    chi = L ** 2 * (m2 - mean_abs_m ** 2)

    tau_int, window, acf = integrated_autocorrelation_time(abs_m)
    naive_err = np.std(abs_m) / np.sqrt(len(abs_m))
    corrected_err = naive_err * np.sqrt(max(2.0 * tau_int, 1.0))
    chi_err = L ** 2 * 2.0 * mean_abs_m * corrected_err

    return {
        'order_param': float(mean_abs_m),
        'order_err': float(corrected_err),
        'susceptibility': float(chi),
        'susceptibility_err': float(chi_err),
        'tau_int': float(tau_int),
        'tau_window': int(window),
        'acf': acf[:300],
    }


def build_phi4_kernel(J, *, L_value=None):
    L_value = int(L if L_value is None else L_value)
    client = LatticeDiffusionClient('phi4', diffusion, fwd, L_value, device=device)
    context = GaussianPBCContext(J, L_value, ns, h=h)
    if not context.check_valid_t_diff(t):
        raise ValueError(f'Invalid Gaussian context for J={J:.3f} at t={t:.3f}.')
    aggregator = FFTGaussianAggregator(J, L_value, ns, h=h, client=client, device=device)
    kernel = FixedDiffusionTimeKernel.from_clients({'phi4': client}, aggregator, context, master_seed=42)
    return kernel, client


def scan_schedule_for_J(J):
    J = float(J)
    in_critical_window = CRITICAL_J_WINDOW[0] <= J <= CRITICAL_J_WINDOW[1]
    n_sweeps = int(CRITICAL_N_SWEEPS if in_critical_window else N_SWEEPS)
    burnin = int(round(BURNIN * n_sweeps / N_SWEEPS))
    return {
        'critical_region': bool(in_critical_window),
        'n_sweeps': n_sweeps,
        'burnin': burnin,
        'record_interval': int(RECORD_INTERVAL),
    }


def run_ggpa_chain(
    J,
    *,
    seed,
    L_value=None,
    n_sweeps=None,
    burnin=None,
    record_interval=None,
):
    L_value = int(L if L_value is None else L_value)
    if n_sweeps is None or burnin is None or record_interval is None:
        schedule = scan_schedule_for_J(J)
        if n_sweeps is None:
            n_sweeps = schedule['n_sweeps']
        if burnin is None:
            burnin = schedule['burnin']
        if record_interval is None:
            record_interval = schedule['record_interval']

    n_sweeps = int(n_sweeps)
    burnin = int(burnin)
    record_interval = int(record_interval)

    kernel, client = build_phi4_kernel(float(J), L_value=L_value)
    rng = np.random.default_rng(seed)
    state = State(s=rng.choice([-1.0, 1.0], size=(L_value, L_value)).astype(np.float64))

    mags = []
    t0 = time.time()
    for sweep in range(n_sweeps):
        state, _ = kernel.step(state, t, compute_reduced_potential=False)
        if sweep % record_interval == 0:
            mags.append(float(np.mean(client.current_x)))

    mags = np.asarray(mags, dtype=np.float64)
    burn_idx = burnin // record_interval
    if burn_idx >= len(mags):
        raise ValueError('Burn-in leaves no recorded frames. Reduce burnin or record_interval.')

    obs = compute_observables_from_history(mags[burn_idx:], L_value)
    obs.update(
        {
            'J': float(J),
            'L': int(L_value),
            'elapsed': float(time.time() - t0),
            'n_sweeps': int(n_sweeps),
            'burnin': int(burnin),
            'record_interval': int(record_interval),
            'critical_region': bool(scan_schedule_for_J(J)['critical_region']),
            'magnetizations': mags,
            'final_config': np.asarray(client.current_x, dtype=np.float64).copy(),
        }
    )
    return obs


def run_quick_ggpa_chain(J, *, seed):
    return run_ggpa_chain(J, seed=seed)

## Final-Frame Snapshots at $L=32$

We first run three separate GG-PA chains at `L=32` with the same fixed sweep count and show only the **final frame** for:

- `J = 0.10`
- `J = 0.436`
- `J = 0.80`

This section is meant as a compact visual illustration of the low-coupling, near-critical, and high-coupling regimes. For each final frame we also print the single-frame order parameter

$$
|m| = \left| \frac{1}{L^2} \sum_i \phi_i \right|.
$$

In [ ]:
SNAPSHOT_L = 32
SNAPSHOT_J_VALUES = [0.10, 0.436, 0.80]
SNAPSHOT_SWEEPS = 2000
SNAPSHOT_BURNIN = 0
SNAPSHOT_RECORD_INTERVAL = 10

snapshot_results = []
for idx, J_snapshot in enumerate(SNAPSHOT_J_VALUES):
    obs = run_ggpa_chain(
        J_snapshot,
        seed=SEED_BASE + 1000 + idx,
        L_value=SNAPSHOT_L,
        n_sweeps=SNAPSHOT_SWEEPS,
        burnin=SNAPSHOT_BURNIN,
        record_interval=SNAPSHOT_RECORD_INTERVAL,
    )
    snapshot_results.append(obs)

vmax = max(float(np.max(np.abs(obs['final_config']))) for obs in snapshot_results)
fig, axes = plt.subplots(1, len(snapshot_results), figsize=(13.2, 4.2), constrained_layout=True)

for ax, obs in zip(axes, snapshot_results):
    final_frame = obs['final_config']
    final_order_param = float(abs(np.mean(final_frame)))
    im = ax.imshow(final_frame, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.set_title(f'J={obs["J"]:.3f}\n|m|={final_order_param:.3f}')
    ax.set_xticks([])
    ax.set_yticks([])
    print(
        f'J={obs["J"]:.3f}  '
        f'final-frame |m|={final_order_param:.4f}  '
        f'elapsed={obs["elapsed"]:.1f}s'
    )

cbar = fig.colorbar(im, ax=axes, shrink=0.88, pad=0.02)
cbar.set_label(r'$\phi_i$')
fig.suptitle(
    rf'2D $\phi^4$ final frames from GG-PA ($L={SNAPSHOT_L}$, $t={t:.2f}$, fixed {SNAPSHOT_SWEEPS} sweeps)',
    fontsize=13,
    y=1.04,
)
plt.show()

## Quick Run: $L=8$ J-Scan at $h=0$

The cell below performs a lightweight GG-PA scan over `J` at fixed `t=0.10`.

For each coupling value we run one short chain, discard an initial burn-in segment, and compute:

- the order parameter $\langle |m| \rangle$
- the susceptibility $\chi = L^2 (\langle m^2 \rangle - \langle |m| \rangle^2)$

This is intentionally a **quick qualitative scan**. Outside the pseudo-critical window we keep the original quick setting `N_SWEEPS = 800`, but for the denser region around `J \approx 0.44` we now use `3000` sweeps automatically, with burn-in scaled by the same fraction as before.

If you want an even sharper peak or smoother error bars, the first knobs to increase are:

- `CRITICAL_N_SWEEPS`
- the density of `J_values` near the peak
- the lattice size `L`

**This is a quick demo, a precise one should use more sweeps with Replica Exchange**

In [ ]:
results = []
for idx, J in enumerate(J_values):
    obs = run_quick_ggpa_chain(J, seed=SEED_BASE + idx)
    results.append(obs)
    region_tag = 'critical' if obs['critical_region'] else 'regular'
    print(
        f'J={J:.3f}  '
        f'region={region_tag}  '
        f'sweeps={obs["n_sweeps"]}  '
        f'<|m|>={obs["order_param"]:.4f} ± {obs["order_err"]:.4f}  '
        f'chi={obs["susceptibility"]:.3f} ± {obs["susceptibility_err"]:.3f}  '
        f'tau_int={obs["tau_int"]:.2f}  '
        f'elapsed={obs["elapsed"]:.1f}s'
    )

scan = {
    'J': np.array([r['J'] for r in results]),
    'order_param': np.array([r['order_param'] for r in results]),
    'order_err': np.array([r['order_err'] for r in results]),
    'susceptibility': np.array([r['susceptibility'] for r in results]),
    'susceptibility_err': np.array([r['susceptibility_err'] for r in results]),
    'tau_int': np.array([r['tau_int'] for r in results]),
}

peak_idx = int(np.argmax(scan['susceptibility']))
J_peak = float(scan['J'][peak_idx])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharex=True)

axes[0].errorbar(
    scan['J'],
    scan['order_param'],
    yerr=scan['order_err'],
    fmt='o-',
    color='#1f77b4',
    lw=1.8,
    ms=5,
    capsize=3,
)
axes[0].axvline(J_peak, color='gray', ls='--', lw=1)
axes[0].axvspan(CRITICAL_J_WINDOW[0], CRITICAL_J_WINDOW[1], color='#f59e0b', alpha=0.10)
axes[0].set_xlabel('J')
axes[0].set_ylabel(r'$\langle |m| \rangle$')
axes[0].set_title(r'Quick GG-PA scan: order parameter')

axes[1].errorbar(
    scan['J'],
    scan['susceptibility'],
    yerr=scan['susceptibility_err'],
    fmt='o-',
    color='#d62728',
    lw=1.8,
    ms=5,
    capsize=3,
)
axes[1].axvline(J_peak, color='gray', ls='--', lw=1, label=rf'peak at $J \approx {J_peak:.3f}$')
axes[1].axvspan(CRITICAL_J_WINDOW[0], CRITICAL_J_WINDOW[1], color='#f59e0b', alpha=0.10, label='3000-sweep window')
axes[1].set_xlabel('J')
axes[1].set_ylabel(r'$\chi$')
axes[1].set_title(r'Quick GG-PA scan: susceptibility')
axes[1].legend(fontsize=9)

fig.suptitle(rf'2D $\phi^4$ quick scan with GG-PA ($L={L}$, $h={h:.0f}$, $t={t:.2f}$)', fontsize=13, y=1.03)
plt.tight_layout()
plt.show()

print(f'\nPseudo-critical coupling from the quick susceptibility peak: J* = {J_peak:.3f}')
print(f'Max susceptibility in this scan: chi(J*) = {scan["susceptibility"][peak_idx]:.3f}')